# Principal Component Analysis (PCA) — From Scratch

PCA is the most widely used **dimensionality-reduction** technique in machine learning and data analysis.
It transforms a dataset of possibly correlated variables into a set of linearly uncorrelated variables
called **principal components**, ranked by the amount of variance they capture.

## Why Dimensionality Reduction?

Real-world datasets often have dozens, hundreds, or even thousands of features.
High dimensionality brings several problems:

| Problem | Description |
|---|---|
| **Curse of dimensionality** | As dimensions grow, data becomes sparse and distances between points converge, making algorithms less effective. |
| **Overfitting** | More features → more parameters → higher risk of fitting noise instead of signal. |
| **Computational cost** | Training time and memory grow with the number of features. |
| **Visualization** | We cannot directly visualize data with more than 3 dimensions. |

PCA addresses all of these by projecting the data onto a **lower-dimensional subspace** while
retaining as much **variance** (information) as possible.

## Roadmap

1. Key concepts: variance, covariance, eigenvectors.
2. The PCA algorithm, step by step.
3. Pure-Python implementations of every building block.
4. Worked example on a small dataset.
5. How to choose the number of components.
6. Limitations of PCA.

---
## 1. Key Concepts

### 1.1 Variance as Information

In PCA the guiding principle is simple: **directions with the most variance carry the most information**.

If a feature has zero variance it is constant across all samples — it tells us nothing useful.
Conversely, a direction along which the data is spread out is informative because it helps
distinguish one sample from another.

Variance of a single feature $x$:

$$\text{Var}(x) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

### 1.2 Covariance

Covariance measures how two features move together:

$$\text{Cov}(x, y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

- $\text{Cov}(x,y) > 0$: $x$ and $y$ tend to increase together.
- $\text{Cov}(x,y) < 0$: when one increases the other decreases.
- $\text{Cov}(x,y) = 0$: no **linear** relationship.

For a dataset with $d$ features, all pairwise covariances are stored in the **covariance matrix** $\Sigma$
(a $d \times d$ symmetric matrix).

### 1.3 Principal Components

A principal component is a new axis (direction) in feature space defined by an **eigenvector** of the
covariance matrix. The corresponding **eigenvalue** tells us how much variance is captured along
that direction.

$$\Sigma \mathbf{v} = \lambda \mathbf{v}$$

- $\mathbf{v}$: eigenvector (direction of a principal component).
- $\lambda$: eigenvalue (variance captured along that direction).

The eigenvectors are orthogonal to each other, meaning the principal components are uncorrelated.

---
## 2. The PCA Algorithm — Step by Step

Given an $n \times d$ data matrix $X$ (n samples, d features):

1. **Standardize** each feature to zero mean and unit variance.
2. **Compute the covariance matrix** $\Sigma$ of the standardized data.
3. **Compute eigenvalues and eigenvectors** of $\Sigma$.
4. **Sort** eigenvectors by their eigenvalues in descending order.
5. **Select** the top $k$ eigenvectors (the $k$ principal components).
6. **Transform** the data: $Z = X_{\text{std}} \cdot W_k$ where $W_k$ is the $d \times k$ matrix of selected eigenvectors.

---
## 3. Pure-Python Building Blocks

We implement every helper function from scratch.
The only place we rely on NumPy is `numpy.linalg.eig` for eigendecomposition,
since a robust, numerically stable eigensolver from scratch is impractical.

### 3.1 Mean and Standard Deviation

In [ ]:
def column_means(X):
    """Compute the mean of each column (feature) in a 2-D list."""
    n = len(X)
    d = len(X[0])
    means = [0.0] * d
    for row in X:
        for j in range(d):
            means[j] += row[j]
    return [m / n for m in means]


def column_stds(X, means):
    """Compute the (sample) standard deviation of each column."""
    n = len(X)
    d = len(X[0])
    var = [0.0] * d
    for row in X:
        for j in range(d):
            var[j] += (row[j] - means[j]) ** 2
    return [(v / (n - 1)) ** 0.5 for v in var]


# Quick sanity check
sample = [[1, 2], [3, 4], [5, 6]]
m = column_means(sample)
s = column_stds(sample, m)
print(f'Means: {m}')
print(f'Stds:  {s}')

### 3.2 Standardize the Data

Standardization (z-score normalization) converts each feature to zero mean and unit variance:

$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$

This is critical for PCA because it is sensitive to the scale of features.
Without standardization a feature measured in kilometers would dominate one measured in centimeters,
simply because its numerical variance is larger.

In [ ]:
def standardize(X):
    """Return a standardized copy of X (2-D list) and the means/stds used."""
    means = column_means(X)
    stds = column_stds(X, means)
    X_std = []
    for row in X:
        new_row = [(row[j] - means[j]) / stds[j] for j in range(len(row))]
        X_std.append(new_row)
    return X_std, means, stds


X_std, _, _ = standardize(sample)
print('Standardized:')
for row in X_std:
    print([round(v, 4) for v in row])

### 3.3 Covariance Matrix — From Scratch

For an $n \times d$ standardized matrix $Z$, the covariance matrix is:

$$\Sigma_{jk} = \frac{1}{n-1}\sum_{i=1}^{n} z_{ij}\, z_{ik}$$

Since the data is already zero-mean after standardization, we do not need to subtract the mean again.

In [ ]:
def covariance_matrix(X_std):
    """Compute the covariance matrix of a standardized 2-D list."""
    n = len(X_std)
    d = len(X_std[0])
    cov = [[0.0] * d for _ in range(d)]
    for j in range(d):
        for k in range(j, d):
            s = 0.0
            for i in range(n):
                s += X_std[i][j] * X_std[i][k]
            cov[j][k] = s / (n - 1)
            cov[k][j] = cov[j][k]  # symmetric
    return cov


cov = covariance_matrix(X_std)
print('Covariance matrix:')
for row in cov:
    print([round(v, 4) for v in row])

### 3.4 Eigendecomposition

We need to solve $\Sigma \mathbf{v} = \lambda \mathbf{v}$ for all eigenvalue–eigenvector pairs.

Writing a numerically stable eigensolver from scratch (e.g., QR algorithm) is beyond the scope of this
tutorial, so we use `numpy.linalg.eig` for this single step. Everything else remains pure Python.

In [ ]:
import numpy as np

def eigen_decomposition(cov_matrix):
    """Wrapper around numpy's eigensolver. Returns (eigenvalues, eigenvectors).
    Each eigenvector is a list (column of the returned matrix)."""
    cov_np = np.array(cov_matrix)
    eigenvalues, eigenvectors = np.linalg.eig(cov_np)
    eigenvalues = eigenvalues.real.tolist()
    eigenvectors = eigenvectors.real.T.tolist()  # each row is an eigenvector
    return eigenvalues, eigenvectors


eigenvalues, eigenvectors = eigen_decomposition(cov)
print('Eigenvalues:', [round(v, 4) for v in eigenvalues])
print('Eigenvectors:')
for ev in eigenvectors:
    print([round(v, 4) for v in ev])

### 3.5 Sort and Select Components

In [ ]:
def sort_eigenpairs(eigenvalues, eigenvectors):
    """Sort eigenpairs by eigenvalue in descending order."""
    pairs = list(zip(eigenvalues, eigenvectors))
    pairs.sort(key=lambda p: p[0], reverse=True)
    sorted_vals = [p[0] for p in pairs]
    sorted_vecs = [p[1] for p in pairs]
    return sorted_vals, sorted_vecs


def explained_variance_ratio(eigenvalues):
    """Fraction of total variance explained by each component."""
    total = sum(eigenvalues)
    return [ev / total for ev in eigenvalues]


sorted_vals, sorted_vecs = sort_eigenpairs(eigenvalues, eigenvectors)
evr = explained_variance_ratio(sorted_vals)
print('Sorted eigenvalues:', [round(v, 4) for v in sorted_vals])
print('Explained variance ratio:', [round(v, 4) for v in evr])

### 3.6 Transform the Data

Projection onto the top $k$ principal components:

$$Z = X_{\text{std}} \cdot W_k$$

where $W_k$ is a $d \times k$ matrix whose columns are the top-$k$ eigenvectors.

In [ ]:
def transform(X_std, eigenvectors_top_k):
    """Project standardized data onto the selected principal components.
    eigenvectors_top_k: list of k eigenvectors (each a list of length d)."""
    n = len(X_std)
    k = len(eigenvectors_top_k)
    d = len(X_std[0])
    Z = []
    for i in range(n):
        row = []
        for c in range(k):
            val = 0.0
            for j in range(d):
                val += X_std[i][j] * eigenvectors_top_k[c][j]
            row.append(val)
        Z.append(row)
    return Z

---
## 4. Putting It All Together — the `pca` Function

In [ ]:
def pca(X, k):
    """Full PCA pipeline.
    Parameters
    ----------
    X : list of lists (n x d)
    k : number of principal components to keep

    Returns
    -------
    Z : transformed data (n x k)
    sorted_vals : all eigenvalues (descending)
    sorted_vecs : all eigenvectors (descending by eigenvalue)
    evr : explained variance ratio for each component
    """
    # Step 1: Standardize
    X_std, means, stds = standardize(X)

    # Step 2: Covariance matrix
    cov = covariance_matrix(X_std)

    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = eigen_decomposition(cov)

    # Step 4: Sort
    sorted_vals, sorted_vecs = sort_eigenpairs(eigenvalues, eigenvectors)

    # Step 5: Explained variance
    evr = explained_variance_ratio(sorted_vals)

    # Step 6: Select top k and transform
    top_k_vecs = sorted_vecs[:k]
    Z = transform(X_std, top_k_vecs)

    return Z, sorted_vals, sorted_vecs, evr

---
## 5. Worked Example

We use the classic **Iris dataset** (4 features, 150 samples, 3 classes) and reduce it to 2 dimensions
for visualization.

### 5.1 Load the Data

In [ ]:
import csv
import os

# Try loading from the project's data directory
iris_paths = [
    os.path.join('..', '13.Naive_Bayes', 'iris.csv'),
    os.path.join('..', '1.Finding_and_Reading_Data', 'IRIS.csv'),
]

dataset = []
labels = []
loaded = False

for path in iris_paths:
    if os.path.exists(path):
        with open(path, 'r') as f:
            reader = csv.reader(f)
            header = next(reader)
            for row in reader:
                if len(row) < 5:
                    continue
                features = [float(row[i]) for i in range(4)]
                dataset.append(features)
                labels.append(row[4].strip())
        loaded = True
        print(f'Loaded {len(dataset)} samples from {path}')
        break

if not loaded:
    # Fallback: generate a synthetic 4-feature dataset
    import random
    random.seed(42)
    class_names = ['A', 'B', 'C']
    centers = [[5.0, 3.4, 1.5, 0.2],
               [5.9, 2.8, 4.3, 1.3],
               [6.6, 3.0, 5.6, 2.0]]
    for ci, center in enumerate(centers):
        for _ in range(50):
            row = [c + random.gauss(0, 0.4) for c in center]
            dataset.append(row)
            labels.append(class_names[ci])
    print(f'Generated synthetic dataset with {len(dataset)} samples')

print(f'Features: {len(dataset[0])}, Samples: {len(dataset)}')
print(f'First 3 rows: {dataset[:3]}')
print(f'Unique labels: {sorted(set(labels))}')

### 5.2 Apply PCA (reduce 4D → 2D)

In [ ]:
Z, eigenvalues_sorted, eigenvectors_sorted, evr = pca(dataset, k=2)

print('Eigenvalues (sorted):', [round(v, 4) for v in eigenvalues_sorted])
print('Explained variance ratio:', [round(v, 4) for v in evr])
print(f'\nTotal variance captured by first 2 PCs: {sum(evr[:2]):.2%}')
print(f'\nFirst 5 transformed samples:')
for row in Z[:5]:
    print([round(v, 4) for v in row])

### 5.3 Visualize the 2-D Projection

In [ ]:
import matplotlib.pyplot as plt

unique_labels = sorted(set(labels))
colors = ['#e74c3c', '#2ecc71', '#3498db']
markers = ['o', 's', '^']

plt.figure(figsize=(8, 6))
for idx, label in enumerate(unique_labels):
    xs = [Z[i][0] for i in range(len(Z)) if labels[i] == label]
    ys = [Z[i][1] for i in range(len(Z)) if labels[i] == label]
    plt.scatter(xs, ys, c=colors[idx % len(colors)],
                marker=markers[idx % len(markers)],
                label=label, alpha=0.7, edgecolors='k', linewidths=0.5)

plt.xlabel(f'PC 1 ({evr[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC 2 ({evr[1]:.1%} variance)', fontsize=12)
plt.title('PCA — 2-D Projection', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Even with just 2 out of 4 original dimensions, the three classes are clearly separated.
This demonstrates PCA's power: most of the discriminative information was concentrated in
the first two principal components.

---
## 6. How to Choose $k$ — The Number of Components

There is no single "right" answer, but common strategies include:

1. **Cumulative explained variance threshold** — pick the smallest $k$ such that the cumulative
   explained variance exceeds a threshold (e.g., 95%).
2. **Scree plot / elbow method** — plot eigenvalues and look for an "elbow" where they drop off.
3. **Domain knowledge** — in some applications (e.g., image compression) the desired dimensionality
   is dictated by storage or speed constraints.

### 6.1 Cumulative Explained Variance

In [ ]:
def cumulative_variance(evr):
    """Return the cumulative explained variance ratio."""
    cum = []
    running = 0.0
    for v in evr:
        running += v
        cum.append(running)
    return cum


cum_var = cumulative_variance(evr)

print('Component | Explained Var | Cumulative Var')
print('-' * 46)
for i in range(len(evr)):
    print(f'   PC {i+1}    |    {evr[i]:.4f}     |    {cum_var[i]:.4f}')

### 6.2 Scree Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

components = list(range(1, len(evr) + 1))

# Left: Individual explained variance
ax1.bar(components, evr, color='#3498db', edgecolor='k', alpha=0.8)
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('Scree Plot', fontsize=14)
ax1.set_xticks(components)

# Right: Cumulative explained variance
ax2.plot(components, cum_var, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax2.axhline(y=0.95, color='gray', linestyle='--', label='95% threshold')
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax2.set_title('Cumulative Explained Variance', fontsize=14)
ax2.set_xticks(components)
ax2.set_ylim([0, 1.05])
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

### 6.3 Selecting $k$ Programmatically

In [ ]:
def select_k(evr, threshold=0.95):
    """Return the smallest k such that cumulative explained variance >= threshold."""
    cum = 0.0
    for k, v in enumerate(evr, start=1):
        cum += v
        if cum >= threshold:
            return k
    return len(evr)


k_95 = select_k(evr, threshold=0.95)
k_99 = select_k(evr, threshold=0.99)
print(f'Components needed for >=95% variance: {k_95}')
print(f'Components needed for >=99% variance: {k_99}')

---
## 7. Verifying Against NumPy

Let us compare our pure-Python results with NumPy's built-in covariance and eigendecomposition
to make sure our implementation is correct.

In [ ]:
X_np = np.array(dataset)
X_np_std = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0, ddof=1)

cov_np = np.cov(X_np_std, rowvar=False)
vals_np, vecs_np = np.linalg.eig(cov_np)

order = np.argsort(vals_np)[::-1]
vals_np = vals_np[order]
vecs_np = vecs_np[:, order]

evr_np = vals_np / vals_np.sum()

print('NumPy eigenvalues:', np.round(vals_np, 4))
print('Ours  eigenvalues:', [round(v, 4) for v in eigenvalues_sorted])
print()
print('NumPy explained var ratio:', np.round(evr_np, 4))
print('Ours  explained var ratio:', [round(v, 4) for v in evr])

---
## 8. Limitations of PCA

PCA is powerful but not a silver bullet. Keep these limitations in mind:

| Limitation | Explanation |
|---|---|
| **Linear only** | PCA finds linear combinations of features. If the underlying structure is non-linear (e.g., a spiral), PCA will fail. Consider kernel PCA or t-SNE for non-linear cases. |
| **Sensitive to scaling** | Features must be standardized first. Without this, features with large magnitudes will dominate the principal components. |
| **Components are not interpretable** | Each principal component is a mixture of all original features. You lose the ability to say "feature X is important". |
| **Variance ≠ usefulness** | The direction of maximum variance may not be the most discriminative for classification. Linear Discriminant Analysis (LDA) is a supervised alternative. |
| **Outlier sensitivity** | Outliers inflate variance and can distort the principal components. Robust PCA variants exist for this. |
| **Assumes orthogonal components** | The constraint that components must be orthogonal can be limiting in some cases. Independent Component Analysis (ICA) relaxes this. |

---
## 9. Summary

In this notebook we implemented **PCA from scratch** in pure Python:

1. **Standardization**: zero mean and unit variance for each feature.
2. **Covariance matrix**: computed element-by-element using the definition.
3. **Eigendecomposition**: used `numpy.linalg.eig` (the only NumPy dependency).
4. **Sorting and selection**: pick the top-$k$ eigenvectors.
5. **Projection**: dot product of standardized data with the eigenvector matrix.

### Key Takeaways

- PCA reduces dimensionality by projecting data onto the directions of **maximum variance**.
- The **covariance matrix** encodes all pairwise linear relationships between features.
- **Eigenvalues** tell us how much variance each component captures; **eigenvectors** give the directions.
- Always **standardize** before applying PCA.
- Use the **cumulative explained variance** or a **scree plot** to decide how many components to keep.
- PCA is **linear** and **unsupervised** — it does not use class labels.